In [1]:
!pip install neo4j pandas

In [2]:
from neo4j import GraphDatabase
import pandas as pd
import os

In [3]:
#neo4j+s://2b1f2d19.databases.neo4j.io

In [3]:
from neo4j import GraphDatabase

URI = "neo4j+s://2b1f2d19.databases.neo4j.io"
USERNAME = "2b1f2d19"
PASSWORD = "dH--Ad36MBlpKWcaOkezZCCn_2Angf6qH90RjgPMS8U"

driver = GraphDatabase.driver(
    URI,
    auth=(USERNAME, PASSWORD)
)

driver.verify_connectivity()

print("Connected successfully!")

Connected successfully!


## Constructing KG from a Table data

In [4]:
data = {
    "faculty": [
        "Dr. Anu",
        "Dr. Ravi",
        "Dr. Manu",
        "Dr. Anu",
        "Dr. Manu"
    ],
    "department": [
        "CSE",
        "ECE",
        "CSE",
        "CSE",
        "CSE"
    ],
    "research_area": [
        "Knowledge Graphs",
        "Machine Learning",
        "Bioinformatics",
        "Bioinformatics",
        "Natural Language Processing"
    ],
    "application": [
        "Bioinformatics",
        "Healthcare",
        "Drug Discovery",
        "Healthcare",
        "Text Analytics"
    ],
    "course": [
        "Artificial Intelligence",
        "Machine Learning",
        "Data Science",
        "Knowledge Representation",
        "Natural Language Processing"
    ]
}

df = pd.DataFrame(data)

df

,faculty,department,research_area,application,course
0,Dr. Anu,CSE,Knowledge Graphs,Bioinformatics,Artificial Intelligence
1,Dr. Ravi,ECE,Machine Learning,Healthcare,Machine Learning
2,Dr. Manu,CSE,Bioinformatics,Drug Discovery,Data Science
3,Dr. Anu,CSE,Bioinformatics,Healthcare,Knowledge Representation
4,Dr. Manu,CSE,Natural Language Processing,Text Analytics,Natural Language Processing


In [5]:
constraints = [
    """
    CREATE CONSTRAINT faculty_name IF NOT EXISTS
    FOR (f:Faculty)
    REQUIRE f.name IS UNIQUE
    """,

    """
    CREATE CONSTRAINT department_name IF NOT EXISTS
    FOR (d:Department)
    REQUIRE d.name IS UNIQUE
    """,

    """
    CREATE CONSTRAINT research_area_name IF NOT EXISTS
    FOR (r:ResearchArea)
    REQUIRE r.name IS UNIQUE
    """,

    """
    CREATE CONSTRAINT application_name IF NOT EXISTS
    FOR (a:Application)
    REQUIRE a.name IS UNIQUE
    """,

    """
    CREATE CONSTRAINT course_name IF NOT EXISTS
    FOR (c:Course)
    REQUIRE c.name IS UNIQUE
    """
]

with driver.session() as session:
    for query in constraints:
        session.run(query)

In [6]:
query = """
MERGE (f:Faculty {name: $faculty})

MERGE (d:Department {name: $department})
MERGE (f)-[:WORKS_IN]->(d)

MERGE (r:ResearchArea {name: $research_area})
MERGE (f)-[:RESEARCHES]->(r)

MERGE (a:Application {name: $application})
MERGE (r)-[:APPLIED_IN]->(a)

MERGE (c:Course {name: $course})
MERGE (f)-[:TEACHES]->(c)
"""

In [7]:
with driver.session() as session:

    for _, row in df.iterrows():

        session.run(
            query,
            faculty=row["faculty"],
            department=row["department"],
            research_area=row["research_area"],
            application=row["application"],
            course=row["course"]
        )

print("Knowledge Graph constructed!")

Knowledge Graph constructed!


## Constructing KG from Text

In [1]:
!pip install -U transformers accelerate bitsandbytes


In [3]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype="auto"
)

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [4]:
text = """
Insulin is a hormone produced by beta cells in the pancreas.
Insulin regulates blood glucose levels by promoting glucose uptake
in muscle and adipose tissues.

Diabetes mellitus is a metabolic disorder associated with
impaired insulin production or insulin action.

Machine learning techniques are increasingly used to predict
diabetes risk using clinical and biochemical data.
"""

In [5]:
prompt = f"""
You are an expert in knowledge graph construction.

Extract entities and relationships from the following text.

Instructions:
1. Identify important entities.
2. Assign an appropriate entity type.
3. Identify explicit relationships between entities.
4. Use concise, meaningful relationship names.
5. Do not invent information that is not present in the text.
6. Return ONLY valid JSON. Do not include markdown or explanations.

Required JSON format:

{{
  "entities": [
    {{"name": "Insulin", "type": "Hormone"}}
  ],
  "relationships": [
    {{
      "source": "Insulin",
      "relation": "PRODUCED_BY",
      "target": "Beta cells"
    }}
  ]
}}

Text:
{text}
"""

print(prompt)


You are an expert in knowledge graph construction.

Extract entities and relationships from the following text.

Instructions:
1. Identify important entities.
2. Assign an appropriate entity type.
3. Identify explicit relationships between entities.
4. Use concise, meaningful relationship names.
5. Do not invent information that is not present in the text.
6. Return ONLY valid JSON. Do not include markdown or explanations.

Required JSON format:

{
  "entities": [
    {"name": "Insulin", "type": "Hormone"}
  ],
  "relationships": [
    {
      "source": "Insulin",
      "relation": "PRODUCED_BY",
      "target": "Beta cells"
    }
  ]
}

Text:

Insulin is a hormone produced by beta cells in the pancreas.
Insulin regulates blood glucose levels by promoting glucose uptake
in muscle and adipose tissues.

Diabetes mellitus is a metabolic disorder associated with
impaired insulin production or insulin action.

Machine learning techniques are increasingly used to predict
diabetes risk using

In [6]:
messages = [
    {"role": "user", "content": prompt}
]

input_text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(
    input_text,
    return_tensors="pt"
).to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=1200,
        do_sample=False
    )

generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

response = tokenizer.decode(
    generated_tokens,
    skip_special_tokens=True
)

print(response)

{
  "entities": [
    {"name": "Insulin", "type": "Hormone"},
    {"name": "Beta cells", "type": "Cell Type"},
    {"name": "Pancreas", "type": "Organ"},
    {"name": "Muscle", "type": "Tissue"},
    {"name": "Adipose tissue", "type": "Tissue"},
    {"name": "Diabetes mellitus", "type": "Disease"},
    {"name": "Clinical and biochemical data", "type": "Data Type"}
  ],
  "relationships": [
    {
      "source": "Insulin",
      "relation": "PRODUCED_BY",
      "target": "Beta cells"
    },
    {
      "source": "Insulin",
      "relation": "REGULATES",
      "target": "Blood glucose levels"
    },
    {
      "source": "Insulin",
      "relation": "INDUCES",
      "target": "Glucose uptake in muscle and adipose tissues"
    },
    {
      "source": "Diabetes mellitus",
      "relation": "ASSOCIATED_WITH",
      "target": "Impaired insulin production or insulin action"
    },
    {
      "source": "Machine learning techniques",
      "relation": "USED_FOR",
      "target": "Predicting d

In [7]:
import json

start = response.find("{")
end = response.rfind("}") + 1

if start == -1 or end == 0:
    raise ValueError("No JSON object found in model response")

kg_data = json.loads(response[start:end])

print("ENTITIES")
for entity in kg_data["entities"]:
    print(entity)

print("\nRELATIONSHIPS")
for relation in kg_data["relationships"]:
    print(relation)

ENTITIES
{'name': 'Insulin', 'type': 'Hormone'}
{'name': 'Beta cells', 'type': 'Cell Type'}
{'name': 'Pancreas', 'type': 'Organ'}
{'name': 'Muscle', 'type': 'Tissue'}
{'name': 'Adipose tissue', 'type': 'Tissue'}
{'name': 'Diabetes mellitus', 'type': 'Disease'}
{'name': 'Clinical and biochemical data', 'type': 'Data Type'}

RELATIONSHIPS
{'source': 'Insulin', 'relation': 'PRODUCED_BY', 'target': 'Beta cells'}
{'source': 'Insulin', 'relation': 'REGULATES', 'target': 'Blood glucose levels'}
{'source': 'Insulin', 'relation': 'INDUCES', 'target': 'Glucose uptake in muscle and adipose tissues'}
{'source': 'Diabetes mellitus', 'relation': 'ASSOCIATED_WITH', 'target': 'Impaired insulin production or insulin action'}
{'source': 'Machine learning techniques', 'relation': 'USED_FOR', 'target': 'Predicting diabetes risk'}
{'source': 'Machine learning techniques', 'relation': 'USING', 'target': 'Clinical and biochemical data'}


In [8]:
import neo4j
from getpass import getpass
from neo4j import GraphDatabase

NEO4J_URI =  "neo4j+s://2b1f2d19.databases.neo4j.io"
NEO4J_USERNAME = "2b1f2d19"
NEO4J_PASSWORD = "dH--Ad36MBlpKWcaOkezZCCn_2Angf6qH90RjgPMS8U"
NEO4J_DATABASE = "2b1f2d19"

driver = GraphDatabase.driver(
    NEO4J_URI,
    auth=(NEO4J_USERNAME, NEO4J_PASSWORD)
)

driver.verify_connectivity()

print("Connected to Neo4j successfully!")

Connected to Neo4j successfully!


In [9]:
def insert_entities(tx, entities):

    for entity in entities:

        query = """
        MERGE (e:Entity {name: $name})
        SET e.type = $type
        """

        tx.run(
            query,
            name=entity["name"],
            type=entity["type"]
        )


with driver.session(database=NEO4J_DATABASE) as session:
    session.execute_write(
        insert_entities,
        kg_data["entities"]
    )

print("Entities inserted successfully!")

Entities inserted successfully!


In [10]:
def insert_relationships(tx, relationships):

    for rel in relationships:

        query = """
        MATCH (s:Entity {name: $source})
        MATCH (t:Entity {name: $target})
        MERGE (s)-[r:RELATED_TO {type: $relation}]->(t)
        """

        tx.run(
            query,
            source=rel["source"],
            target=rel["target"],
            relation=rel["relation"]
        )


with driver.session(database=NEO4J_DATABASE) as session:
    session.execute_write(
        insert_relationships,
        kg_data["relationships"]
    )

print("Relationships inserted successfully!")

Relationships inserted successfully!


In [11]:
with driver.session(database=NEO4J_DATABASE) as session:

    result = session.run("""
        MATCH (s:Entity)-[r:RELATED_TO]->(t:Entity)
        RETURN s, r, t
    """)

    records = list(result)

    print(f"Number of relationships: {len(records)}")

Number of relationships: 1
